In [1]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
# %matplotlib inline
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

In [2]:
import bayesopinf_mcmc_check as mcmc_opinf

In [3]:
r = 3

# prob = mcmc_opinf.build_problem()
prob = mcmc_opinf.build_problem(noise=0.005, T_train=6.0, K=300, n_traj=5,
                                deriv="lpr", window=None, poly=3,
                                T_test=12.0)   # train on [0, 6], test on (6, 12]
r_, d = prob["mu"].shape

results = []
for i in range(r_):
    s = mcmc_opinf.sample_row(prob, i)
    res = mcmc_opinf.compare_row(s, prob["mu"][i], prob["Sigma"][i])
    results.append(res)
    print(f"\nrow {i}:  min ESS = {res['ess'].min():.0f} (2nd-moment ESS = {res['ess2']:.0f}),  max R-hat = {res['rhat'].max():.3f}")
    print(f"  mean z-scores: max|z| = {np.abs(res['z']).max():.2f}, "
            f"mean z^2 = {np.mean(res['z']**2):.2f}  (expect ~1)")
    tol = 2 * np.sqrt(d / res['ess2'])
    print(f"  whitened cov eigenvalues in [{res['eig'].min():.3f}, {res['eig'].max():.3f}]  "
            f"(expect within ~1 ± {tol:.3f})")
    print(f"  rel. Frobenius cov error = {res['rel_frob']:.3f}")
    print(f"  KL(MCMC || analytic) = {res['kl']:.4f}  (finite-sample baseline ~ {res['kl_expected']:.4f})")

[lpr] d = 10 params/row, K = 1500 snapshots, cond(D^T D + Gamma) = 1.38e+03, windows = [43, 33, 31, 51, 41] pts (0.86 time units)

row 0:  min ESS = 3564 (2nd-moment ESS = 4314),  max R-hat = 1.001
  mean z-scores: max|z| = 1.50, mean z^2 = 0.63  (expect ~1)
  whitened cov eigenvalues in [0.935, 1.068]  (expect within ~1 ± 0.096)
  rel. Frobenius cov error = 0.032
  KL(MCMC || analytic) = 0.0041  (finite-sample baseline ~ 0.0064)

row 1:  min ESS = 3574 (2nd-moment ESS = 4299),  max R-hat = 1.001
  mean z-scores: max|z| = 2.38, mean z^2 = 1.83  (expect ~1)
  whitened cov eigenvalues in [0.940, 1.066]  (expect within ~1 ± 0.096)
  rel. Frobenius cov error = 0.026
  KL(MCMC || analytic) = 0.0056  (finite-sample baseline ~ 0.0064)

row 2:  min ESS = 3174 (2nd-moment ESS = 4118),  max R-hat = 1.002
  mean z-scores: max|z| = 1.75, mean z^2 = 1.30  (expect ~1)
  whitened cov eigenvalues in [0.913, 1.074]  (expect within ~1 ± 0.099)
  rel. Frobenius cov error = 0.047
  KL(MCMC || analytic) = 

In [ ]:
# marginal QQ plots for a few entries of each row
fig, axes = plt.subplots(r_, 3, figsize=(10, 3 * r_))
names = ["c"] + [f"A{k}" for k in range(r)] + [f"H{k}" for k in range(d - 1 - r)]
for i, res in enumerate(results):
    for col, j in enumerate([0, 1 + i, d - 1]):
        x = np.sort(res["flat"][:, j])
        p = (np.arange(len(x)) + 0.5) / len(x)
        q_th = stats.norm.ppf(p, prob["mu"][i, j], np.sqrt(prob["Sigma"][i, j, j]))
        ax = axes[i, col]
        ax.plot(q_th, x, ".", ms=1.5, alpha=0.5)
        lo, hi = q_th[0], q_th[-1]
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set_title(f"row {i}, {names[j]}", fontsize=9)
        ax.set_xlabel("analytical quantile", fontsize=8)
        ax.set_ylabel("MCMC quantile", fontsize=8)
fig.tight_layout()
# fig.savefig("qq_marginals.png", dpi=300)
fig.savefig("qq_marginals.pdf")

In [ ]:
# push-forward bands: training window [0, T_train] + extrapolation window (T_train, T_test]
n_pf = 500
idx = rng.choice(results[0]["flat"].shape[0], n_pf, replace=False)
O_mcmc = np.stack([np.stack([res["flat"][k] for res in results]) for k in idx])
O_exact = np.stack([
    np.stack([rng.multivariate_normal(prob["mu"][i], prob["Sigma"][i]) for i in range(r_)])
    for _ in range(n_pf)
])
t, T_train = prob["t_full"], prob["T_train"]
Y_mcmc = mcmc_opinf.pushforward(O_mcmc, t, prob["q0"])
Y_exact = mcmc_opinf.pushforward(O_exact, t, prob["q0"])
print(f"\npush-forward: {len(Y_mcmc)}/{n_pf} MCMC and {len(Y_exact)}/{n_pf} analytical rollouts stayed finite")
for k in range(r_):
    b_m = np.percentile(Y_mcmc[:, k], [2.5, 97.5], axis=0)
    b_e = np.percentile(Y_exact[:, k], [2.5, 97.5], axis=0)
    gap = np.abs(b_m - b_e).max() / (b_e[1] - b_e[0]).max()
    print(f"  q{k}: max band-edge gap / max band width = {gap:.3f}")

# prediction skill vs. the noiseless truth, training window vs. extrapolation window
scores = {lab: mcmc_opinf.score_prediction(Y, t, prob["Q_true"], T_train)
          for lab, Y in [("analytical", Y_exact), ("NUTS", Y_mcmc)]}
hdr = (f"\n{'posterior':>10s} | {'window':>6s} | {'rel err':>7s} | {'95% cov':>7s} | "
       f"{'band width':>10s} | {'CRPS':>7s}")
print(hdr); print("-" * len(hdr))
for lab, sc in scores.items():
    for w in ["train", "test"]:
        if w in sc:
            v = sc[w]
            print(f"{lab:>10s} | {w:>6s} | {v['rel_err']:7.4f} | {v['cov']:7.2f} | "
                  f"{v['width']:10.4f} | {v['crps']:7.4f}")

fig, axes = plt.subplots(r_, 1, figsize=(8, 2.4 * r_), sharex=True)
for k in range(r_):
    ax = axes[k]
    for Y, lab, col in [(Y_exact, "analytical Gaussian", "C0"), (Y_mcmc, "NUTS", "C1")]:
        lo, med, hi = np.percentile(Y[:, k], [2.5, 50, 97.5], axis=0)
        ax.fill_between(t, lo, hi, color=col, alpha=0.25, label=f"{lab} 95%")
        ax.plot(t, med, color=col, lw=1)
    ax.plot(t, prob["Q_true"][k], "k--", lw=0.8, label="truth")
    ax.plot(prob["t"], prob["Q"][k], "k.", ms=1, label="training data")
    ax.plot(t[t > T_train], prob["Q_test"][k], ".", color="0.6", ms=1, label="test data")
    ax.axvline(T_train, color="0.5", lw=0.8, ls=":")
    ax.set_ylabel(f"$\\hat q_{k}$")
axes[0].legend(fontsize=7, loc="upper right", ncol=3)
axes[-1].set_xlabel("t  (dotted: end of training window)")
fig.tight_layout()
# fig.savefig("pushforward_bands.png", dpi=300)
fig.savefig("pushforward_bands.pdf")

In [ ]:
# relative error of the median prediction over time: growth past T_train
fig, ax = plt.subplots(figsize=(7, 2.6))
for (lab, sc), col in zip(scores.items(), ["C0", "C1"]):
    ax.semilogy(t, sc["err_t"], color=col, lw=1, label=lab)
ax.axvline(T_train, color="0.5", lw=0.8, ls=":")
ax.set_xlabel("t"); ax.set_ylabel("rel. error of median")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("extrapolation_error.pdf")
print("saved qq_marginals.pdf, pushforward_bands.pdf, extrapolation_error.pdf")